In [1]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import importlib
import json
import sys
from IPython.display import Image, display

model_paths = [
    "/kaggle/input/models/leonidtikhanov/pinn-model/pytorch/default/10",
    "",
]

for p in model_paths:
    if Path(p).exists():
        sys.path.insert(0, p)
        break

import pinn_model
pinn_model = importlib.reload(pinn_model)

print(pinn_model.__file__)
print("run_experiment:", hasattr(pinn_model, "run_experiment"))


/kaggle/input/models/leonidtikhanov/pinn-model/pytorch/default/10/pinn_model.py
run_experiment: True


In [2]:
print("torch version:", torch.__version__)
print("cuda available:", torch.cuda.is_available())

if torch.cuda.is_available():
    device = "cuda"
    print("gpu:", torch.cuda.get_device_name(0))
else:
    device = "cpu"

work_dir = Path("/kaggle/working")
if not work_dir.exists():
    work_dir = Path(".")

out_dir = work_dir / "results_exp_17_burgers1d_adam_lr_check"
out_dir.mkdir(parents=True, exist_ok=True)

print("device:", device)
print("work_dir:", work_dir)
print("out_dir:", out_dir)


torch version: 2.10.0+cu128
cuda available: True


gpu: Tesla T4
device: cuda
work_dir: /kaggle/working
out_dir: /kaggle/working/results_exp_17_burgers1d_adam_lr_check


In [3]:
nu_values = [0.001, 0.002, 0.005]
dtype_values = ["fp32", "fp64"]
seed_values = [0]

adam_values = [
    {
        "adam_name": "adam5e4",
        "lr_adam": 5e-4,
    },
    {
        "adam_name": "adam2e4",
        "lr_adam": 2e-4,
    },
    {
        "adam_name": "adam1e4",
        "lr_adam": 1e-4,
    },
]

lbfgs_values = [
    {
        "lbfgs_name": "lbfgs1",
        "lbfgs_lr": 1.0,
    },
    {
        "lbfgs_name": "lbfgs05",
        "lbfgs_lr": 0.5,
    },
]

runs = []
i = 1
for nu in nu_values:
    for adam in adam_values:
        for lbfgs in lbfgs_values:
            for dtype in dtype_values:
                for seed in seed_values:
                    row = {
                        "run_id": i,
                        "nu": nu,
                        "dtype": dtype,
                        "seed": seed,
                        "adam_name": adam["adam_name"],
                        "lbfgs_name": lbfgs["lbfgs_name"],
                        "variant": f"{adam['adam_name']}_{lbfgs['lbfgs_name']}_steps2000_inner5_resample200",
                        "adam_steps": 3000,
                        "lbfgs_steps": 2000,
                        "lr_adam": adam["lr_adam"],
                        "lbfgs_max_iter": 5,
                        "lbfgs_max_eval": None,
                        "lbfgs_lr": lbfgs["lbfgs_lr"],
                        "lbfgs_history_size": 100,
                        "lbfgs_tolerance_grad": 1e-8,
                        "lbfgs_tolerance_change": 1e-10,
                        "lbfgs_line_search_fn": "strong_wolfe",
                        "resample_every": 200,
                        "hid_size": 256,
                        "num_layers": 4,
                        "init_gain": None,
                        "n_collocation": 10000,
                        "n_ic": 800,
                        "n_bc": 800,
                    }
                    runs.append(row)
                    i += 1

pd.DataFrame(runs)[["run_id", "nu", "variant", "adam_name", "lbfgs_name", "dtype", "seed", "lr_adam", "lbfgs_lr", "hid_size", "num_layers", "adam_steps", "lbfgs_steps"]]


,run_id,nu,variant,adam_name,lbfgs_name,dtype,seed,lr_adam,lbfgs_lr,hid_size,num_layers,adam_steps,lbfgs_steps
0,1,0.001,adam5e4_lbfgs1_steps2000_inner5_resample200,adam5e4,lbfgs1,fp32,0,0.0005,1.0,256,4,3000,2000
1,2,0.001,adam5e4_lbfgs1_steps2000_inner5_resample200,adam5e4,lbfgs1,fp64,0,0.0005,1.0,256,4,3000,2000
2,3,0.001,adam5e4_lbfgs05_steps2000_inner5_resample200,adam5e4,lbfgs05,fp32,0,0.0005,0.5,256,4,3000,2000
3,4,0.001,adam5e4_lbfgs05_steps2000_inner5_resample200,adam5e4,lbfgs05,fp64,0,0.0005,0.5,256,4,3000,2000
4,5,0.001,adam2e4_lbfgs1_steps2000_inner5_resample200,adam2e4,lbfgs1,fp32,0,0.0002,1.0,256,4,3000,2000
5,6,0.001,adam2e4_lbfgs1_steps2000_inner5_resample200,adam2e4,lbfgs1,fp64,0,0.0002,1.0,256,4,3000,2000
6,7,0.001,adam2e4_lbfgs05_steps2000_inner5_resample200,adam2e4,lbfgs05,fp32,0,0.0002,0.5,256,4,3000,2000
7,8,0.001,adam2e4_lbfgs05_steps2000_inner5_resample200,adam2e4,lbfgs05,fp64,0,0.0002,0.5,256,4,3000,2000
8,9,0.001,adam1e4_lbfgs1_steps2000_inner5_resample200,adam1e4,lbfgs1,fp32,0,0.0001,1.0,256,4,3000,2000
9,10,0.001,adam1e4_lbfgs1_steps2000_inner5_resample200,adam1e4,lbfgs1,fp64,0,0.0001,1.0,256,4,3000,2000


In [4]:
base_config = {
    "task_name": "burgers1d",
    "dtype": "fp32",
    "seed": 0,
    "device": device,
    "nu": nu_values[0],
    "hid_size": 256,
    "num_layers": 4,
    "init_gain": None,
    "n_collocation": 10000,
    "n_ic": 800,
    "n_bc": 800,
    "adam_steps": 3000,
    "lbfgs_steps": 2000,
    "lr_adam": 5e-4,
    "resample_every": 200,
    "use_adam": True,
    "use_lbfgs": True,
    "lbfgs_tolerance_grad": 1e-8,
    "lbfgs_tolerance_change": 1e-10,
    "lbfgs_history_size": 100,
    "lbfgs_lr": 1.0,
    "lbfgs_max_iter": 5,
    "lbfgs_max_eval": None,
    "lbfgs_line_search_fn": "strong_wolfe",
    "log_dir": str(out_dir / "runs" / "burgers1d_tmp"),
}

pd.DataFrame(runs).groupby(["nu", "variant", "dtype"]).size()


nu     variant                                       dtype
0.001  adam1e4_lbfgs05_steps2000_inner5_resample200  fp32     1
                                                     fp64     1
       adam1e4_lbfgs1_steps2000_inner5_resample200   fp32     1
                                                     fp64     1
       adam2e4_lbfgs05_steps2000_inner5_resample200  fp32     1
                                                     fp64     1
       adam2e4_lbfgs1_steps2000_inner5_resample200   fp32     1
                                                     fp64     1
       adam5e4_lbfgs05_steps2000_inner5_resample200  fp32     1
                                                     fp64     1
       adam5e4_lbfgs1_steps2000_inner5_resample200   fp32     1
                                                     fp64     1
0.002  adam1e4_lbfgs05_steps2000_inner5_resample200  fp32     1
                                                     fp64     1
       adam1e4_lbfgs1_steps2000_inner5_resamp

In [5]:
all_summaries = []
all_histories = {}

for run in runs:
    config = base_config.copy()
    config.update(run)
    config["use_adam"] = config["adam_steps"] > 0
    config["use_lbfgs"] = config["lbfgs_steps"] > 0
    nu_tag = str(run["nu"]).replace(".", "p")
    name = f"exp17_r{run['run_id']:03d}_nu{nu_tag}_{run['variant']}_{run['dtype']}_s{run['seed']}"
    config["log_dir"] = str(out_dir / "runs" / name)

    run_dir = Path(config["log_dir"])
    summary_file = run_dir / "summary.json"
    metrics_file = run_dir / "metrics.csv"

    if summary_file.exists() and metrics_file.exists():
        with open(summary_file) as f:
            summary = json.load(f)
        history = pd.read_csv(metrics_file)
    else:
        history, summary = pinn_model.run_experiment(config)

    best = history.loc[history["l2_error"].idxmin()]
    summary["run_id"] = run["run_id"]
    summary["run_name"] = name
    summary["variant"] = run["variant"]
    summary["adam_name"] = run["adam_name"]
    summary["lbfgs_name"] = run["lbfgs_name"]
    summary["nu"] = run["nu"]
    summary["dtype"] = run["dtype"]
    summary["seed"] = run["seed"]
    summary["adam_steps_config"] = run["adam_steps"]
    summary["lr_adam"] = run["lr_adam"]
    summary["lbfgs_steps_config"] = run["lbfgs_steps"]
    summary["lbfgs_max_iter"] = run["lbfgs_max_iter"]
    summary["lbfgs_lr"] = run["lbfgs_lr"]
    summary["resample_every"] = run["resample_every"]
    summary["n_collocation"] = run["n_collocation"]
    summary["n_ic"] = run["n_ic"]
    summary["n_bc"] = run["n_bc"]
    summary["hid_size"] = run["hid_size"]
    summary["num_layers"] = run["num_layers"]
    summary["init_gain"] = run["init_gain"]
    summary["best_l2_error"] = float(best["l2_error"])
    summary["best_step"] = int(best["step"])
    summary["final_l2_error"] = float(history["l2_error"].iloc[-1])
    summary["elapsed_time"] = float(history["time_sec"].iloc[-1])
    all_summaries.append(summary)
    all_histories[name] = history

    print(run["run_id"], run["variant"], "nu", run["nu"], run["dtype"], "seed", run["seed"], "final", summary["final_l2_error"], "best", summary["best_l2_error"], "time", summary["elapsed_time"])

summary_df = pd.DataFrame(all_summaries)
summary_path = out_dir / "exp_17_burgers1d_adam_lr_check_summary.csv"
summary_df.to_csv(summary_path, index=False)
summary_path


/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:330.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


1 adam5e4_lbfgs1_steps2000_inner5_resample200 nu 0.001 fp32 seed 0 final 0.27088141441345215 best 0.27088141441345215 time 686.0835032463074


2 adam5e4_lbfgs1_steps2000_inner5_resample200 nu 0.001 fp64 seed 0 final 0.1003156039928465 best 0.09958194150329346 time 5011.353064537048


3 adam5e4_lbfgs05_steps2000_inner5_resample200 nu 0.001 fp32 seed 0 final 0.23826481401920319 best 0.23362486064434052 time 662.4213654994965


4 adam5e4_lbfgs05_steps2000_inner5_resample200 nu 0.001 fp64 seed 0 final 0.0910408607097001 best 0.07859920721555647 time 4988.230743646622


...

11 adam1e4_lbfgs05_steps2000_inner5_resample200 nu 0.001 fp32 seed 0 final 0.17678102850914001 best 0.17678102850914001 time 650.5304620265961


12 adam1e4_lbfgs05_steps2000_inner5_resample200 nu 0.001 fp64 seed 0 final 0.21475275158232335 best 0.18577982827130268 time 4998.292836427689


13 adam5e4_lbfgs1_steps2000_inner5_resample200 nu 0.002 fp32 seed 0 final 0.050614576786756516 best 0.047775086015462875 time 677.3033442497253


14 adam5e4_lbfgs1_steps2000_inner5_resample200 nu 0.002 fp64 seed 0 final 0.050917320703828754 best 0.048379421062698866 time 5015.097866296768


15 adam5e4_lbfgs05_steps2000_inner5_resample200 nu 0.002 fp32 seed 0 final 0.04712007939815521 best 0.04702397808432579 time 677.5922255516052


In [ ]:
cols = [
    "run_id",
    "variant",
    "adam_name",
    "lbfgs_name",
    "nu",
    "dtype",
    "seed",
    "hid_size",
    "num_layers",
    "n_collocation",
    "n_ic",
    "n_bc",
    "adam_steps_config",
    "lr_adam",
    "lbfgs_steps_config",
    "lbfgs_max_iter",
    "lbfgs_lr",
    "resample_every",
    "final_l2_error",
    "best_l2_error",
    "best_step",
    "elapsed_time",
]

summary_df[cols].sort_values(["nu", "adam_name", "lbfgs_name", "dtype", "seed"])


In [ ]:
grouped = summary_df.groupby(["variant", "adam_name", "lbfgs_name", "nu", "dtype"])[["best_l2_error", "final_l2_error", "elapsed_time"]].agg(["mean", "std", "min", "max"])
grouped.columns = ["_".join(c).strip("_") for c in grouped.columns]
grouped = grouped.reset_index()
grouped.to_csv(out_dir / "exp_17_burgers1d_adam_lr_check_grouped.csv", index=False)

ratio = grouped.pivot_table(index=["variant", "adam_name", "lbfgs_name", "nu"], columns="dtype", values="best_l2_error_mean").reset_index()
if "fp32" in ratio.columns and "fp64" in ratio.columns:
    ratio["fp64_over_fp32_best"] = ratio["fp64"] / ratio["fp32"]
ratio.to_csv(out_dir / "exp_17_burgers1d_adam_lr_check_fp64_ratio.csv", index=False)
ratio


In [ ]:
for nu in sorted(summary_df["nu"].unique()):
    fig, ax = plt.subplots(1, 2, figsize=(14, 4))
    for name, hist in all_histories.items():
        row = summary_df[summary_df["run_name"] == name].iloc[0]
        if row["nu"] != nu:
            continue
        label = f"{row['adam_name']} {row['lbfgs_name']} {row['dtype']}"
        ax[0].plot(hist["step"], hist["total_loss"], label=label)
        ax[1].plot(hist["step"], hist["l2_error"], label=label)
    ax[0].set_yscale("log")
    ax[1].set_yscale("log")
    ax[0].set_xlabel("step")
    ax[1].set_xlabel("step")
    ax[0].set_ylabel("total_loss")
    ax[1].set_ylabel("l2_error")
    ax[0].grid(True)
    ax[1].grid(True)
    ax[0].legend(fontsize=7)
    ax[1].legend(fontsize=7)
    fig.suptitle(f"burgers1d nu={nu}, steps2000_inner5_resample200")
    fig.tight_layout()
    plt.show()


In [ ]:
for row in summary_df.sort_values(["nu", "adam_name", "lbfgs_name", "dtype", "seed"]).itertuples():
    p = Path(row.log_dir) / "curves.png"
    if p.exists():
        print(row.run_name)
        display(Image(filename=str(p)))


In [ ]:
for row in summary_df.sort_values(["nu", "adam_name", "lbfgs_name", "dtype", "seed"]).itertuples():
    p = Path(row.log_dir) / "solution_t1.png"
    if p.exists():
        print(row.run_name)
        display(Image(filename=str(p)))


In [ ]:
bad = summary_df[summary_df["best_l2_error"] > 0.5]
bad_path = out_dir / "exp_17_burgers1d_adam_lr_check_bad_runs.csv"
bad[cols].to_csv(bad_path, index=False)
bad[cols].sort_values("best_l2_error", ascending=False)
